# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [106]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve the PRICE_DATA directory from environment variables
price_data_dir = os.getenv("PRICE_DATA")

# Ensure PRICE_DATA is set
if not price_data_dir:
    raise ValueError("Environment variable PRICE_DATA is not set. Please execute 01_materials/labs/2_data_engineering.ipynb to create this data set.")

# Print the directory to confirm it's loaded correctly
print("PRICE_DATA directory:", price_data_dir)



PRICE_DATA directory: ../../05_src/data/prices/


In [107]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [108]:
import os
from glob import glob
# Correct path to match the location of files
price_data_dir = "/Users/admin/production/05_src/data/prices"

# Check if the directory exists
if os.path.exists(price_data_dir):
    print("Directory exists:", price_data_dir)
    print("Contents of the directory:", os.listdir(price_data_dir))

Directory exists: /Users/admin/production/05_src/data/prices
Contents of the directory: ['CTAS', 'WELL', 'VZ', 'AMZN', 'CNP', 'RCL', 'CAT', 'TFC', 'AAPL', 'PANW', 'PM', 'KHC', 'TEL', 'CMCSA', 'GRMN', 'ANET', 'BRO', 'CAH', 'TROW', 'PODD', 'FSLR', 'PFE', 'REG', 'TFX', 'APTV', 'AAL', 'CDW', 'MAR', 'VRSN', 'FITB', 'KMI', 'UBER', 'SPGI', 'ALLE', 'SYK', 'MMC', 'MPWR', 'DXCM', 'PEP', 'FRT', 'SNPS', 'PLD', 'GOOG', 'MMM', 'EMN', 'NXPI', 'PCG', 'BF.B', 'AMT', 'ADI', 'MAA', 'HWM', 'MAS', 'FTV', 'CAG', 'MGM', 'CNC', 'VLO', 'UPS', 'CME', 'NWS', 'EMR', 'OKE', 'SYY', 'GILD', 'SNA', 'MS', 'BIO', 'PTC', 'AXON', 'WMT', 'HUBB', 'ULTA', 'DG', 'CVX', 'NCLH', 'UHS', 'GPN', 'DLR', 'MO', 'CPT', 'TXN', 'KLAC', 'INTC', 'BK', 'NI', 'TTWO', 'HCA', 'GS', 'TMUS', 'BLK', 'CDNS', 'LOW', 'LIN', 'MA', 'HII', 'CZR', 'DFS', 'PRU', 'BX', 'KVUE', 'CTSH', 'SHW', 'MCHP', 'VTR', 'HIG', 'MU', 'TXT', 'ACGL', 'APD', 'HD', 'FIS', 'IPG', 'NDSN', 'UNP', 'AMAT', 'LLY', 'WBD', 'FOX', 'JNPR', 'NTAP', 'FICO', 'MSI', 'AKAM', 'SBAC', 'WA

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [110]:
import dask.dataframe as dd

# Convert paths to absolute paths
parquet_files = [os.path.abspath(f) for f in parquet_files]

# Read Parquet files using Dask
ddf = dd.read_parquet(parquet_files, engine="pyarrow")

# Display the first few rows to verify the data
print(ddf.head())

Price                       Date  Adj Close      Close       High        Low  \
Ticker                                                                         
A      2000-01-03 00:00:00+00:00  43.382843  51.502148  56.464592  48.193848   
A      2000-01-04 00:00:00+00:00  40.068878  47.567955  49.266811  46.316166   
A      2000-01-05 00:00:00+00:00  37.583401  44.617310  47.567955  43.141991   
A      2000-01-06 00:00:00+00:00  36.152378  42.918453  44.349072  41.577251   
A      2000-01-07 00:00:00+00:00  39.165062  46.494991  47.165592  42.203148   

Price        Open     Volume  Year  
Ticker                              
A       56.330471  4674353.0  2000  
A       48.730328  4765083.0  2000  
A       47.389126  5758642.0  2000  
A       44.080830  2534434.0  2000  
A       42.247852  2819626.0  2000  


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [114]:
import pandas as pd

# Convert Dask DataFrame to Pandas
pdf = ddf.compute()

# Ensure `Date` is in datetime format
pdf["Date"] = pd.to_datetime(pdf["Date"])

# Sort by Ticker and Date before applying rolling window
pdf = pdf.sort_values(["Ticker", "Date"])

# Print a sample to verify
print("Successfully converted to Pandas.")
print(pdf.head())

Successfully converted to Pandas.
Price                       Date  Adj Close      Close       High        Low  \
Ticker                                                                         
A      2000-01-03 00:00:00+00:00  43.382843  51.502148  56.464592  48.193848   
A      2000-01-04 00:00:00+00:00  40.068878  47.567955  49.266811  46.316166   
A      2000-01-05 00:00:00+00:00  37.583401  44.617310  47.567955  43.141991   
A      2000-01-06 00:00:00+00:00  36.152378  42.918453  44.349072  41.577251   
A      2000-01-07 00:00:00+00:00  39.165062  46.494991  47.165592  42.203148   

Price        Open     Volume  Year  
Ticker                              
A       56.330471  4674353.0  2000  
A       48.730328  4765083.0  2000  
A       47.389126  5758642.0  2000  
A       44.080830  2534434.0  2000  
A       42.247852  2819626.0  2000  


In [116]:
# Compute the 10-day moving average of returns
pdf["returns_moving_avg_10"] = (
    pdf.groupby("Ticker")["returns"]
    .rolling(10, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# Print a sample to verify
print("Successfully added 10-day moving average.")
print(pdf.head())

Successfully added 10-day moving average.
Price                       Date  Adj Close      Close       High        Low  \
Ticker                                                                         
A      2000-01-03 00:00:00+00:00  43.382843  51.502148  56.464592  48.193848   
A      2000-01-04 00:00:00+00:00  40.068878  47.567955  49.266811  46.316166   
A      2000-01-05 00:00:00+00:00  37.583401  44.617310  47.567955  43.141991   
A      2000-01-06 00:00:00+00:00  36.152378  42.918453  44.349072  41.577251   
A      2000-01-07 00:00:00+00:00  39.165062  46.494991  47.165592  42.203148   

Price        Open     Volume  Year  Close_lag_1   returns  \
Ticker                                                      
A       56.330471  4674353.0  2000          NaN       NaN   
A       48.730328  4765083.0  2000    51.502148 -0.076389   
A       47.389126  5758642.0  2000    47.567955 -0.062030   
A       44.080830  2534434.0  2000    44.617310 -0.038076   
A       42.247852  2819626.0  2

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return? NO
+ Would it have been better to do it in Dask? Why?
YES scalability and performanceless overhead
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.